# Qwen3-TTS Voice Clone

Clone any voice with a ~10 second reference audio using [Qwen3-TTS](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-Base).

**How to use:**
1. Click **Runtime → Run all** (or press `Ctrl+Shift+F9`)
2. When prompted, upload a reference audio file
3. Wait for the voice clone to generate — it will play automatically and download to your computer

**Requirements:** T4 GPU runtime (free tier)

## Block 1 — GPU check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Block 2 — Install dependencies

In [ ]:
# Install system-level sox (required by qwen-tts)
!apt-get install -y -q sox > /dev/null 2>&1

# Pin transformers to version where check_model_inputs() decorator works with parentheses
!pip install -q transformers==4.50.0 accelerate torchaudio soundfile librosa sox onnxruntime einops qwen-omni-utils nagisa soynlp 2>/dev/null

# Install qwen packages with --no-deps
!pip install -q --no-deps qwen-tts
!pip install -q --no-deps qwen-asr

print("✅ Dependencies installed")

## Block 3 — Import modules

In [ ]:
import torch
import soundfile as sf
import librosa
import numpy as np
from qwen_tts import Qwen3TTSModel
from qwen_asr import Qwen3ASRModel
from IPython.display import Audio, display
from google.colab import files
import os
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Imports loaded")

## Block 4 — Configure

Edit these settings directly in the cells below, or use the form widgets.

In [ ]:
#@title Voice cloning settings

#@markdown **TTS Model** — 1.7B is higher quality, 0.6B is faster and uses less VRAM
model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-Base" #@param ["Qwen/Qwen3-TTS-12Hz-0.6B-Base", "Qwen/Qwen3-TTS-12Hz-1.7B-Base"]

#@markdown **Text to speak** — what should the cloned voice say?
generated_text = "Hello! This is a test of the Qwen3 voice cloning system. The voice you hear was generated from a short reference audio sample." #@param {type:"string"}

#@markdown ---
print(f"Model: {model_name}")
print(f"Text: {generated_text[:80]}...")

## Block 5 — Upload reference audio

Click "Choose Files" to upload a voice recording (.wav, .mp3, etc.).

**Best results:** 3–10 seconds of clear, single-speaker speech with no background noise.

In [ ]:
REFERENCE_VOICE_FILE = "reference_voice.wav"
OUTPUT_VOICE_FILE = "cloned_voice.wav"

print("📁 Upload a reference audio file...")
uploaded_files = files.upload()

if not uploaded_files:
    raise SystemExit("No file uploaded — re-run this cell to try again.")

original_filename = list(uploaded_files.keys())[0]
os.rename(original_filename, REFERENCE_VOICE_FILE)

# Validate
print("\n🔍 Validating...")
audio_check, sr_check = librosa.load(REFERENCE_VOICE_FILE, sr=None, mono=True)
duration = len(audio_check) / sr_check
rms = np.sqrt(np.mean(audio_check**2))

if duration < 2:
    print(f"⚠️  Only {duration:.1f}s — recommend 3–10s for best results")
elif duration > 30:
    print(f"⚠️  {duration:.1f}s — recommend 3–10s, longer clips may have noise")
else:
    print(f"✅ Duration: {duration:.1f}s")

if rms < 0.001:
    print("⚠️  Audio appears silent — upload a file with audible speech")
else:
    print(f"✅ Audio level OK")

print("\n📻 Reference audio:")
display(Audio(REFERENCE_VOICE_FILE))

## Block 6 — Auto-transcribe reference audio

Uses Qwen3-ASR to transcribe the reference audio. This transcript is used as `ref_text` for accurate speaker embedding — the model needs to know what words are spoken to align text-to-audio features.

In [ ]:
asr_model_name = "Qwen/Qwen3-ASR-0.6B"
asr_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Loading ASR model: {asr_model_name}")
asr_model = Qwen3ASRModel.from_pretrained(asr_model_name, dtype=asr_dtype, device_map="auto")

print("🎙️ Transcribing reference audio...")
with torch.inference_mode():
    results = asr_model.transcribe(audio=REFERENCE_VOICE_FILE, language=None, return_time_stamps=False)

if not results or not results[0].text.strip():
    raise ValueError("ASR returned empty transcription — upload a clearer audio file")

reference_text = results[0].text.strip()
print(f"✅ Transcribed: {reference_text}")

# Free VRAM
del asr_model
torch.cuda.empty_cache()
print("🧹 ASR model unloaded")

## Block 7 — Load TTS model

In [ ]:
if device == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if "1.7B" in model_name and vram_gb < 16:
        print(f"⚠️  VRAM: {vram_gb:.1f}GB — 1.7B model may OOM. Consider switching to 0.6B.")

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Loading TTS model: {model_name}")
model = Qwen3TTSModel.from_pretrained(
    model_name,
    dtype=dtype,
    device_map="auto"
)
print("✅ TTS model loaded")

## Block 8 — Generate cloned voice

In [ ]:
def detect_language(text):
    if not text.strip():
        return "English"
    chinese_chars = sum(1 for c in text if "\u4e00" <= c <= "\u9fff")
    return "Chinese" if chinese_chars / len(text) > 0.3 else "English"

target_language = detect_language(generated_text)

print(f"🎵 Generating voice clone...")
print(f"   Language: {target_language}")
print(f"   Reference: {reference_text}")

start = time.time()

try:
    with torch.inference_mode():
        wavs, sr = model.generate_voice_clone(
            text=generated_text,
            language=target_language,
            ref_audio=REFERENCE_VOICE_FILE,
            ref_text=reference_text
        )
except Exception as e:
    print(f"❌ Voice cloning failed: {e}")
    print("   Try a shorter audio file or switch to the 0.6B model")
    raise

elapsed = time.time() - start
print(f"✅ Done in {elapsed:.1f}s — {len(wavs[0]) / sr:.2f}s of audio")

## Block 9 — Save and download

In [ ]:
sf.write(OUTPUT_VOICE_FILE, wavs[0], sr)
print(f"💾 Saved as {OUTPUT_VOICE_FILE}")

# Play it back
display(Audio(OUTPUT_VOICE_FILE))

# Auto-download
try:
    files.download(OUTPUT_VOICE_FILE)
    print("✅ Downloading...")
except Exception:
    print(f"Manual download: Files panel → {OUTPUT_VOICE_FILE}")